# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the FAIR² [Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant pandas matplotlib

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import json

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}\n")
print(metadata.description)

## 2. Data Overview

Let's review available record sets, their `@id`s, and associated fields. We will use these IDs to extract and manipulate records.

The dataset may contain several record sets (tables or collections of records) and each has fields/columns. We will list these using the Croissant metadata interface.

In [ ]:
print("Available Record Sets:")
all_record_sets = list(metadata.record_sets)
for rset in all_record_sets:
    print(f"- Name: {rset.name} (@id: {rset.id})")
    print("  Fields (columns):")
    for field in rset.fields:
        desc = getattr(field, 'description', None)
        col_info = f"  - {field.name} (@id: {field.id})"
        if desc:
            col_info += f" | {desc}"
        print(col_info)
    print("")

### Example: Print first 2 records from the main record set
Now, let's print the first two records for each available record set to get an idea of the structure.

In [ ]:
for rset in all_record_sets:
    print(f"Records from Record Set: {rset.name} (@id: {rset.id})")
    for i, rec in enumerate(dataset.records(record_set=rset.id)):
        print(json.dumps(rec, indent=2))
        if i >= 1:
            break
    print("")

## 3. Data Extraction
Based on the overview above, let's extract the main data table(s) into DataFrames for analysis. We will specify the record set `@id`s.  
Replace `<main_record_set_id>` with the wanted record set (usually the one with most fields/records).

In [ ]:
# Build a list of available record set IDs
record_sets_ids = [rset.id for rset in all_record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    print(f"Loading records for record set '{record_set_id}'")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns: {df.columns.tolist()}")
    print(df.head())

# Choose one main record set for further analysis
if len(record_sets_ids) == 0:
    raise Exception('No record sets found in Croissant metadata.')
main_record_set_id = record_sets_ids[0]  # Use the first by default
main_df = dataframes[main_record_set_id]
print(f"\nSelected main record set for analysis: {main_record_set_id}")
print("Columns:", main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)

Let's select a numeric field (column) for basic analysis, such as filtering, normalization, and grouping. You can change the field IDs based on the printed columns above.

We will:
- Filter records by a threshold in a numeric column
- Normalize that column
- Optionally group by a string or categorical field

In [ ]:
# Identify possible numeric fields (those with dtype int or float)
import numpy as np
numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
if not numeric_fields:
    print("No numeric fields detected; attempting to coerce columns to numeric where possible.")
    for col in main_df.columns:
        # Try to convert to numeric, ignore errors
        coerced = pd.to_numeric(main_df[col], errors='coerce')
        if coerced.notnull().any():
            main_df[col] = coerced
            if pd.api.types.is_numeric_dtype(main_df[col]):
                numeric_fields.append(col)
print(f"Numeric fields available for EDA: {numeric_fields}")

# Pick the first available numeric field for demonstration
if not numeric_fields:
    raise Exception("No numeric columns available in this dataset!")

numeric_field = numeric_fields[0]
print(f"Using '{numeric_field}' as the main numeric column.")

# Pick a reasonable threshold (e.g., mean or median)
if main_df[numeric_field].notnull().sum() == 0:
    print(f"Warning: Column '{numeric_field}' is empty!")
threshold = main_df[numeric_field].median() if main_df[numeric_field].notnull().sum() else 0

filtered_df = main_df[main_df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical field (e.g., something with <20 unique values and not numeric)
candidate_group_fields = [col for col in main_df.columns if pd.api.types.is_object_dtype(main_df[col]) and main_df[col].nunique() < 20]
if candidate_group_fields:
    group_field = candidate_group_fields[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization

Visualize the distribution of the selected numeric field, and if grouped, the mean per group.

In [ ]:
# Histogram of the numeric field
plt.figure(figsize=(8, 4))
main_df[numeric_field].dropna().hist(bins=20)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If we did grouping, make a bar plot
if 'grouped_df' in locals():
    plt.figure(figsize=(10, 4))
    grouped_df.plot.bar(x=group_field, y=numeric_field, legend=False)
    plt.title(f"Mean {numeric_field} per {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We loaded and explored the FAIR² mass spectrometry dataset using `mlcroissant`.
- We reviewed its record set(s) and available fields using their `@id`s for robust references.
- We demonstrated extracting records as DataFrames, basic numeric filtering, normalization, and visualization of the main quantitative data field.

For further analyses, you might join with annotation tables, explore additional record sets, or extend filtering/grouping—simply adapt the field and record set `@id` references shown above.

---
This notebook is editable—please adapt columns/IDs and add new EDA or modeling code as required for your domain and analysis goals!